# 🖼️ Image Classification — From Checkpoint to Embedded C

**Renesas RA8P1 Model Zoo — Tutorial 01**

---

This notebook walks through the complete pipeline for deploying an image-classification model
onto a **Renesas RA8P1 MCU** (Cortex-M85 + Ethos-U55 NPU) using the **MERA (RUHMI) compiler**.

```
tf.keras.applications (ImageNet weights from Keras)
        │
        ▼  [TFLiteConverter — Section 2]
  TFLite FP32  (python/model/<name>_FP32.tflite)
        │
        ▼  [post-training quantization — Section 3]
  TFLite INT8  (python/model/<name>_INT8.tflite)
        │
        ▼  [MERA compiler — Section 5]
  C source files  (embedded_c/src_mcu/ or src_mcu_npu/)
        │
        ▼
  Flash to RA8P1 board
```

---

### Models in this Tutorial

| # | Model | Folder | Dataset | Input | Format | Top-1 Acc |
|---|-------|--------|---------|-------|--------|-----------|
| 1 | **MobileNetV1 (0.25×)** | `vision/image_classification/mobilenetv1/` | ImageNet | 224×224 | `keras_app` | 44.70% |
| 2 | **MobileNetV2 (1.0×)** | `vision/image_classification/mobilenetv2/` | ImageNet | 224×224 | `keras_app` | 67.40% |

> 📌 **Select your model** in the **Configuration** cell (cell 4) by setting `SELECTED_MODEL`.

---

### Prerequisites

> ▶️ **Run cell 2** to install all dependencies automatically.

For the MERA compiler (Sections 5 & 6), uncomment and update the wheel path in cell 2
with the path to your Renesas SDK `mera-*.whl` file.

### References
- 📁 [MobileNetV1 README](../vision/image_classification/mobilenetv1/README.md)
- 📁 [MobileNetV2 README](../vision/image_classification/mobilenetv2/README.md)
- 📖 [Renesas Model Zoo](https://github.com/Renesas-Connectivity/Model-zoo)
- 📖 [RA8P1 Product Page](https://www.renesas.com/en/products/ra8p1)
- 📖 [MERA SDK (RUHMI) Docs](https://github.com/renesas/ruhmi-framework-mcu/tree/main/docs)


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Run this cell once to install all required packages into the active kernel.
# Reads requirements.txt from the selected model's python/ folder.
# ─────────────────────────────────────────────────────────────────────────────
import pathlib, os

_nb_dir   = pathlib.Path(os.getcwd())
_repo     = _nb_dir.parent if _nb_dir.name == "tutorials" else _nb_dir
_sentinel = _repo / "vision"
if not _sentinel.exists():
    for _p in _nb_dir.parents:
        if (_p / "vision").exists():
            _repo = _p
            break
    else:
        raise FileNotFoundError(
            f"❌ Could not find the Model-zoo repo root.\n"
            f"   Expected a 'vision/' directory in or above: {_nb_dir}\n"
            f"   Make sure you cloned the full repo and are running this notebook from the 'tutorials/' folder."
        )

#upgrade pip
!pip install --upgrade pip
# ── MERA (RUHMI) compiler wheel ───────────────────────────────────────────────
# Downloads MERA 2.6.0 from the Renesas GitHub repo and installs it.
# Uncomment the three lines below to install:
MERA_WHL = "mera-2.6.0+pkg.4513-cp310-cp310-manylinux_2_27_x86_64.whl"
MERA_URL  = f"https://raw.githubusercontent.com/renesas/ruhmi-framework-mcu/main/install/{MERA_WHL}"
!wget -q --show-progress -O "{MERA_WHL}" "{MERA_URL}" && pip install "{MERA_WHL}"

# # ── Model-specific requirements ───────────────────────────────────────────────
SELECTED_MODEL = "mobilenetv2"   # ← keep in sync with the config cell below
_req = _repo / "vision" / "image_classification" / SELECTED_MODEL / "python" / "requirements.txt"
if not _req.exists():
    raise FileNotFoundError(f"requirements.txt not found: {_req}")
print(f"Installing from: {_req}")
!pip install -r "{_req}"

print("✅ Dependencies installed")

## ⚙️ Setup — Configuration

Set `SELECTED_MODEL` to one of: `"mobilenetv1"`, `"mobilenetv2"`


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

NUM_CALIB_SAMPLES = 200          # calibration samples for INT8 quantization
FORCE_REBUILD     = True        # set True to regenerate models even if they exist

# ─────────────────────────────────────────────────────────────────────────────
import pathlib, os

# Auto-detect repo root from this notebook's location (tutorials/ → parent)
_nb_dir   = pathlib.Path(os.getcwd())
REPO_ROOT = _nb_dir.parent if _nb_dir.name == "tutorials" else _nb_dir
if not (REPO_ROOT / "vision").exists():
    for _p in _nb_dir.parents:
        if (_p / "vision").exists():
            REPO_ROOT = _p
            break
    else:
        raise FileNotFoundError(
            f"❌ Could not find the Model-zoo repo root.\n"
            f"   Expected a 'vision/' directory in or above: {_nb_dir}\n"
            f"   Make sure you cloned the full repo and are running this notebook from the 'tutorials/' folder."
        )

IC_ROOT = REPO_ROOT / "vision" / "image_classification"

print(f"Repo root : {REPO_ROOT}")
print(f"IC root   : {IC_ROOT}")

# ─────────────────────────────────────────────────────────────────────────────
# Model registry — MobileNetV1 and MobileNetV2 (shared ImageNet preprocessing)
# Both models are loaded from tf.keras.applications (no file download needed).
# TFLite files are already included in the repo under python/model/.
# ─────────────────────────────────────────────────────────────────────────────
MODEL_REGISTRY = {
    "mobilenetv1": {
        "display_name"      : "MobileNetV1 (alpha=0.25)",
        "dataset"           : "ImageNet-1k",
        "input_shape"       : (224, 224, 3),
        "num_classes"       : 1000,
        "class_names"       : None,
        "model_dir"         : IC_ROOT / "mobilenetv1",
        "checkpoint_format" : "keras_app",
        "fp32_tflite"       : "mobilenet_v1_FP32.tflite",
        "int8_tflite"       : "mobilenet_v1_INT8.tflite",
        "input_norm"        : "imagenet",   # pixel / 127.5 − 1.0  → [−1, 1]
        "top_k"             : 5,
    },
    "mobilenetv2": {
        "display_name"      : "MobileNetV2 (alpha=1.0)",
        "dataset"           : "ImageNet-1k",
        "input_shape"       : (224, 224, 3),
        "num_classes"       : 1000,
        "class_names"       : None,
        "model_dir"         : IC_ROOT / "mobilenetv2",
        "checkpoint_format" : "keras_app",
        "fp32_tflite"       : "mobilenet_v2_FP32.tflite",
        "int8_tflite"       : "mobilenet_v2_INT8.tflite",
        "input_norm"        : "imagenet",
        "top_k"             : 5,
    },
}

if SELECTED_MODEL not in MODEL_REGISTRY:
    raise ValueError(f"Unknown model '{SELECTED_MODEL}'. Choose: {list(MODEL_REGISTRY)}")

cfg = MODEL_REGISTRY[SELECTED_MODEL]

MODEL_DIR  = cfg["model_dir"]
TFLITE_DIR = MODEL_DIR / "python" / "model"
MERA_DIR   = MODEL_DIR / "python" / "mera_output"
EMBED_DIR  = MODEL_DIR / "embedded_c"

TFLITE_DIR.mkdir(parents=True, exist_ok=True)
MERA_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Model         : {cfg['display_name']}")
print(f"   Format        : {cfg['checkpoint_format']}")
print(f"   Device        : Renesas RA8P1 (Cortex-M85 + Ethos-U55 NPU)")
print(f"   Dataset       : {cfg['dataset']}")
print(f"   Input shape   : {cfg['input_shape']}")
print(f"   Preprocessing : pixel / 127.5 − 1.0  (ImageNet standard)")
print(f"   Classes       : {cfg['num_classes']}")
print(f"   TFLite dir    : {TFLITE_DIR}")
print(f"   MERA output   : {MERA_DIR}")
print(f"   Embedded C    : {EMBED_DIR}")

---
## 📥 Section 1 — Load the Pretrained Checkpoint

Both MobileNetV1 and MobileNetV2 are loaded directly from `tf.keras.applications`
with ImageNet pretrained weights — **no file download required**.

The weights are fetched automatically by Keras on first run and cached locally.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # Force CPU — no GPU needed for conversion

import numpy as np
import tensorflow as tf

print(f"TensorFlow version : {tf.__version__}")
print(f"Loading {cfg['display_name']} from tf.keras.applications ...\n")

if SELECTED_MODEL == "mobilenetv1":
    model = tf.keras.applications.MobileNet(
        input_shape=cfg["input_shape"], alpha=0.25,
        weights="imagenet", include_top=True
    )
elif SELECTED_MODEL == "mobilenetv2":
    model = tf.keras.applications.MobileNetV2(
        input_shape=cfg["input_shape"], alpha=1.0,
        weights="imagenet", include_top=True
    )

print(f"✅ Loaded {cfg['display_name']}")
print(f"   Input  shape : {model.input_shape}")
print(f"   Output shape : {model.output_shape}")
print(f"   Parameters   : {model.count_params():,}")


---
### 🔬 Section 1.1 — Verify Original Keras Model (Baseline Inference)

Before converting to TFLite, run a quick inference using the **original Keras model**
directly. This serves as a **ground-truth baseline** for comparing against the FP32
and INT8 TFLite models later.

| What | Details |
|------|---------|
| **Model** | `tf.keras.applications` (in-memory) |
| **API** | `model.predict()` |
| **Preprocessing** | `pixel / 127.5 − 1.0` → `[−1, 1]` (ImageNet standard) |
| **Output** | Softmax probabilities → top-5 predictions |


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ── Load ImageNet labels ──────────────────────────────────────────────────────
_labels_file = MODEL_DIR / "python" / "utils" / "imagenet_labels.txt"
if _labels_file.exists():
    _raw = _labels_file.read_text().splitlines()
    LABELS = _raw[1:] if len(_raw) == 1001 else _raw
else:
    LABELS = [f"class_{i}" for i in range(1000)]
    print(f"⚠️  Labels file not found at {_labels_file} — using class indices.")

def label_name(idx):
    return LABELS[idx].replace("_", " ") if idx < len(LABELS) else f"class_{idx}"

# ── Load sample images ───────────────────────────────────────────────────────
SAMPLE_DIR = MODEL_DIR / "python" / "sample_images"
SAMPLE_IMAGES = sorted(SAMPLE_DIR.glob("*.jpg")) + sorted(SAMPLE_DIR.glob("*.JPEG")) + sorted(SAMPLE_DIR.glob("*.png"))
if not SAMPLE_IMAGES:
    raise FileNotFoundError(f"No sample images found in {SAMPLE_DIR}")
print(f"Found {len(SAMPLE_IMAGES)} sample images\n")

H, W = cfg["input_shape"][:2]
top_k = cfg["top_k"]

print(f"{'='*60}")
print(f"  Original Keras Model — {cfg['display_name']}")
print(f"{'='*60}\n")

for img_path in tqdm(SAMPLE_IMAGES, desc="Keras inference", unit="img"):
    pil_img   = Image.open(img_path).convert("RGB").resize((W, H))
    test_img  = np.array(pil_img, dtype=np.float32)
    net_input = (test_img / 127.5) - 1.0

    # Keras model inference (ground-truth baseline)
    preds = model.predict(net_input[np.newaxis], verbose=0)[0]
    top_idx = np.argsort(preds)[::-1][:top_k]

    # Show image
    fig, ax = plt.subplots(1, 1, figsize=(3, 3))
    ax.imshow(pil_img)
    ax.axis("off")
    ax.set_title(img_path.name, fontsize=10)
    plt.tight_layout()
    plt.show()

    tqdm.write(f"  Keras top-{top_k}:")
    for rank, i in enumerate(top_idx, 1):
        tqdm.write(f"    {rank}. [{i:>4}] {label_name(i):<30}  {preds[i]*100:.2f}%")
    tqdm.write("")

print(f"✅ Original Keras model verified — predictions look correct.")
print(f"   This serves as the ground-truth baseline for TFLite comparison.")


---
## 🔄 Section 2 — Convert to TFLite FP32

This step produces a **floating-point TFLite flatbuffer** (`.tflite`) — the reference model
with no quantization — which is used as input to the MERA compiler.

Both MobileNetV1 and MobileNetV2 are loaded from `tf.keras.applications`, so conversion
uses `TFLiteConverter.from_keras_model()`. Pre-built FP32 TFLite files are already included
in the repo under `python/model/`, so **this cell will skip conversion** if the file exists.

> 🔍 You can inspect the `.tflite` file with [Netron](https://netron.app).


In [ ]:
fp32_path = TFLITE_DIR / cfg["fp32_tflite"]

if fp32_path.exists() and not FORCE_REBUILD:
    # Pre-built FP32 TFLite already in the repo — use it directly
    print(f"ℹ️  Pre-built FP32 TFLite found — skipping conversion.")
    print(f"   Using: {fp32_path}")
    print(f"   (Set FORCE_REBUILD = True in the config cell to regenerate)")
else:
    if FORCE_REBUILD and fp32_path.exists():
        print(f"🔄  FORCE_REBUILD enabled — regenerating FP32 model ...")
        fp32_path.unlink()
    # Convert from the in-memory Keras model
    if model is None:
        raise RuntimeError("Keras model is None — re-run Section 1.")
    print(f"Converting {cfg['display_name']} → TFLite FP32 …")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    fp32_path.write_bytes(converter.convert())
    print(f"✅ Saved → {fp32_path}")

size_kb = fp32_path.stat().st_size / 1024
print(f"\n   File size : {size_kb:.1f} KB")

interp = tf.lite.Interpreter(model_content=fp32_path.read_bytes())
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print(f"   Input  : name={inp['name']!r}  shape={inp['shape']}  dtype={inp['dtype'].__name__}")
print(f"   Output : name={out['name']!r}  shape={out['shape']}  dtype={out['dtype'].__name__}")


---
## 📉 Section 3 — Post-Training Quantization → TFLite INT8

**Why quantize?** Quantizing weights and activations from `float32` to `int8` gives:
- ~4× reduction in model size (float32 → int8)
- ~2–4× faster inference on MCUs with hardware INT8 support
- Enables the MERA NPU path on RA8xx devices

### How it works
TFLite **post-training quantization** uses a small *representative dataset* (calibration data)
to measure the activation ranges of each layer. It then chooses `scale` and `zero_point`
values so that INT8 values accurately represent the original float range.

```
float_value = (int8_value - zero_point) × scale
int8_value  = round(float_value / scale) + zero_point
```

```
TFLite FP32  +  calibration data  ──►  TFLiteConverter (INT8)  ──►  model_int8.tflite
```

In [ ]:
# ── Calibration dataset generator (ImageNet) ──────────────────────────────────
#
# For best accuracy, use real ImageNet validation images for calibration.
# Download from: https://image-net.org/download.php (registration required).
#
# If no calibration images are available, random arrays will be used instead.
# Random calibration will still produce a working INT8 model, but with
# slightly lower accuracy compared to using real images.
# ─────────────────────────────────────────────────────────────────────────────
import pathlib
from PIL import Image

try:
    from tqdm.auto import tqdm as _tqdm
    _HAS_TQDM = True
except ImportError:
    _HAS_TQDM = False

CALIB_DIR = ""   # ← set this to your ImageNet val dir, e.g. "/path/to/ILSVRC2012_img_val"

H, W = cfg["input_shape"][:2]
USE_RANDOM_CALIB = False

if CALIB_DIR and pathlib.Path(CALIB_DIR).is_dir():
    calib_path = pathlib.Path(CALIB_DIR)
    images = (sorted(calib_path.glob("*.jpg")) +
              sorted(calib_path.glob("*.JPEG")) +
              sorted(calib_path.glob("*.png")))
    if images:
        print(f"   Found {len(images)} images — using {min(NUM_CALIB_SAMPLES, len(images))} for calibration")
    else:
        USE_RANDOM_CALIB = True
        print(f"⚠️  No images found in {calib_path} — falling back to random calibration.")
else:
    USE_RANDOM_CALIB = True
    print(f"⚠️  CALIB_DIR not set — using random arrays for calibration.")
    print(f"   For better accuracy, set CALIB_DIR to your ImageNet validation directory.")

def calib_gen():
    if USE_RANDOM_CALIB:
        rng = np.random.default_rng(42)
        _iter = range(NUM_CALIB_SAMPLES)
        if _HAS_TQDM:
            _iter = _tqdm(_iter, desc="  calibrating (random)", unit="img")
        for _ in _iter:
            arr = rng.uniform(-1.0, 1.0, (1, H, W, 3)).astype(np.float32)
            yield [arr]
    else:
        _items = images[:NUM_CALIB_SAMPLES]
        if _HAS_TQDM:
            _items = _tqdm(_items, desc="  calibrating", unit="img")
        for p in _items:
            img = Image.open(p).convert("RGB").resize((W, H))
            arr = (np.array(img, dtype=np.float32) / 127.5) - 1.0
            yield [arr[np.newaxis]]

print(f"✅ Calibration generator ready  ({NUM_CALIB_SAMPLES} samples, {'random' if USE_RANDOM_CALIB else 'ImageNet'})")

In [ ]:
int8_path = TFLITE_DIR / cfg["int8_tflite"]

if int8_path.exists() and not FORCE_REBUILD:
    # Pre-built INT8 TFLite already in the repo — use it directly
    print(f"ℹ️  Pre-built INT8 TFLite found in repo — skipping PTQ conversion.")
    print(f"   Using: {int8_path}")
    print(f"   (Set FORCE_REBUILD = True in the config cell to regenerate)")
    tflite_int8 = int8_path.read_bytes()

else:
    if FORCE_REBUILD and int8_path.exists():
        print(f"🔄  FORCE_REBUILD enabled — regenerating INT8 model ...")
        int8_path.unlink()
    print(f"Quantizing {cfg['display_name']} → TFLite INT8 …")
    print(f"Calibration samples: {NUM_CALIB_SAMPLES}")

    fmt = cfg["checkpoint_format"]
    if fmt in ("pytorch", "onnx"):
        saved_model_dir = TFLITE_DIR / "onnx2tf_out"
        if not saved_model_dir.exists():
            raise FileNotFoundError(
                f"onnx2tf SavedModel not found at {saved_model_dir}.\nRe-run Section 1."
            )
        converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    else:
        if model is None:
            raise RuntimeError("Keras model is None — re-run Section 1.")
        converter = tf.lite.TFLiteConverter.from_keras_model(model)

    converter.optimizations             = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset    = calib_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type      = tf.int8
    converter.inference_output_type     = tf.int8

    tflite_int8 = converter.convert()
    int8_path.write_bytes(tflite_int8)
    print(f"✅ Saved INT8 TFLite → {int8_path}")

fp32_kb = fp32_path.stat().st_size / 1024
int8_kb  = int8_path.stat().st_size / 1024
print(f"\n   FP32 size : {fp32_kb:.1f} KB")
print(f"   INT8 size : {int8_kb:.1f} KB  ({fp32_kb/int8_kb:.1f}× smaller)")

interp8 = tf.lite.Interpreter(model_content=int8_path.read_bytes())
interp8.allocate_tensors()
inp8 = interp8.get_input_details()[0]
out8 = interp8.get_output_details()[0]
print(f"\n   Input  : dtype={inp8['dtype'].__name__}  quant={inp8['quantization']}")
print(f"   Output : dtype={out8['dtype'].__name__}  quant={out8['quantization']}")


---
## 🔥 Section 4 — Quick Inference Test (TFLite)

Before handing the model to MERA, run a quick sanity check using the TFLite interpreter
directly on a sample image. This confirms the model is working as expected.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ── Load ImageNet labels ──────────────────────────────────────────────────────
_labels_file = MODEL_DIR / "python" / "utils" / "imagenet_labels.txt"
if _labels_file.exists():
    _raw = _labels_file.read_text().splitlines()
    LABELS = _raw[1:] if len(_raw) == 1001 else _raw
else:
    LABELS = [f"class_{i}" for i in range(1000)]
    print(f"⚠️  Labels file not found at {_labels_file} — using class indices.")

def label_name(idx):
    return LABELS[idx].replace("_", " ") if idx < len(LABELS) else f"class_{idx}"

def run_tflite_inference(model_path, input_array):
    """Run single-image inference with a TFLite model. Returns class scores."""
    interp = tf.lite.Interpreter(model_path=str(model_path))
    interp.allocate_tensors()
    inp_det = interp.get_input_details()[0]
    out_det = interp.get_output_details()[0]

    inp_arr = np.expand_dims(input_array, axis=0)
    if inp_det["dtype"] == np.int8:
        scale, zp = inp_det["quantization"]
        inp_arr = (inp_arr / scale + zp).astype(np.int8)
    interp.set_tensor(inp_det["index"], inp_arr)
    interp.invoke()
    out = interp.get_tensor(out_det["index"])[0].astype(np.float32)
    if out_det["dtype"] == np.int8:
        scale, zp = out_det["quantization"]
        out = (out - zp) * scale
    return out

# ── Load sample images ───────────────────────────────────────────────────────
SAMPLE_DIR = MODEL_DIR / "python" / "sample_images"
SAMPLE_IMAGES = sorted(SAMPLE_DIR.glob("*.jpg")) + sorted(SAMPLE_DIR.glob("*.JPEG")) + sorted(SAMPLE_DIR.glob("*.png"))
if not SAMPLE_IMAGES:
    raise FileNotFoundError(f"No sample images found in {SAMPLE_DIR}")
print(f"Found {len(SAMPLE_IMAGES)} sample images: {[p.name for p in SAMPLE_IMAGES]}\n")

H, W = cfg["input_shape"][:2]
top_k = cfg["top_k"]

# ── Run inference on each image and display results ──────────────────────────
for img_path in tqdm(SAMPLE_IMAGES, desc="TFLite FP32/INT8", unit="img"):
    pil_img   = Image.open(img_path).convert("RGB").resize((W, H))
    test_img  = np.array(pil_img, dtype=np.float32)
    net_input = (test_img / 127.5) - 1.0

    scores_fp32 = run_tflite_inference(fp32_path, net_input)
    scores_int8 = run_tflite_inference(int8_path, net_input)

    top_fp32 = np.argsort(scores_fp32)[::-1][:top_k]
    top_int8 = np.argsort(scores_int8)[::-1][:top_k]

    # Show image
    fig, ax = plt.subplots(1, 1, figsize=(3, 3))
    ax.imshow(pil_img)
    ax.axis("off")
    ax.set_title(img_path.name, fontsize=10)
    plt.tight_layout()
    plt.show()

    # Print predictions
    tqdm.write(f"  FP32 top-{top_k}:")
    for rank, i in enumerate(top_fp32, 1):
        tqdm.write(f"    {rank}. [{i:>4}] {label_name(i):<30}  {scores_fp32[i]*100:.2f}%")
    tqdm.write(f"  INT8 top-{top_k}:")
    for rank, i in enumerate(top_int8, 1):
        tqdm.write(f"    {rank}. [{i:>4}] {label_name(i):<30}  {scores_int8[i]*100:.2f}%")
    tqdm.write("")


---
## 🏗️ Section 5 — MERA Compilation

Compiles the model to MCU C-code using the **MERA (RUHMI)** compiler.

---

### 5.1 — NPU Compilation

Compile for **Ethos-U55 NPU** (Requires INT8 model)

```
[CPU+NPU]  mobilenet_vX_INT8.tflite
            │
            ▼
       python mcu_compile.py model_INT8.tflite embedded_c/src_mcu_npu/ --npu
            └── Ethos-U55 NPU C source files
```


In [ ]:
import numpy as np
import pathlib

COMPILE_PY = REPO_ROOT / "ruhmi_tools" / "mcu_compile.py"
NPU_OUT    = EMBED_DIR / "src_mcu_npu"

if not COMPILE_PY.exists():
    raise FileNotFoundError(f"❌ MERA compile script not found: {COMPILE_PY}")
if not int8_path.exists():
    raise FileNotFoundError(f"❌ INT8 TFLite model not found: {int8_path}\n   Run Sections 2–3 first.")

# ── NPU compile (INT8 TFLite → Ethos-U55 NPU C) ────────────────────────────
print("=" * 60)
print("  NPU Compilation")
print("=" * 60)
print(f"  Input : {int8_path}")
!python "{COMPILE_PY}" "{int8_path}" "{NPU_OUT}" --npu

print(f"\n✅ NPU output → {NPU_OUT}")


---
### 5.2 — CPU-Only Compilation

Compile for **Cortex-M CPU (CMSIS-NN)** with `--host-evaluate` to also build
a `py_compute.so` shared library for host-side verification.

```
mobilenet_vX_INT8.tflite
        │
        ▼
python mcu_compile.py model_INT8.tflite src_mcu/ --cpu --host-evaluate
        └── CMSIS-NN C files  →  embedded_c/src_mcu/
        └── py_compute.so     →  host verification library
```

In [ ]:
import shutil, os, subprocess, sys

COMPILE_PY = REPO_ROOT / "ruhmi_tools" / "mcu_compile.py"
CPU_OUT    = EMBED_DIR / "src_mcu"

if not COMPILE_PY.exists():
    raise FileNotFoundError(f"❌ MERA compile script not found: {COMPILE_PY}")
if not int8_path.exists():
    raise FileNotFoundError(f"❌ INT8 TFLite model not found: {int8_path}\n   Run Sections 2–3 first.")

# ── CPU compile (INT8 TFLite → CMSIS-NN C) ───────────────────────────────────
print("=" * 60)
print("  CPU-Only Compilation (with host evaluation)")
print("=" * 60)
print(f"  Input : {int8_path}")

# ⚠️  The CMake host-evaluate build requires a C++ compiler.
env = os.environ.copy()
if not shutil.which("g++"):
    for _ver in ["11", "12", "13", "14"]:
        _gpp = shutil.which(f"g++-{_ver}")
        _gcc = shutil.which(f"gcc-{_ver}")
        if _gpp and _gcc:
            env["CXX"] = _gpp
            env["CC"]  = _gcc
            print(f"  ⚠️  g++ not in PATH — using: CXX={_gpp}, CC={_gcc}")
            break
    else:
        print("  ❌ WARNING: No C++ compiler found! Install: sudo apt install g++-11")

# --host-evaluate builds py_compute.so via CMake so we can verify
# the compiled model on the host before flashing to the board.
result = subprocess.run(
    [sys.executable, str(COMPILE_PY), str(int8_path), str(CPU_OUT), "--cpu", "--host-evaluate"],
    env=env,
)

print(f"\n✅ CPU output → {CPU_OUT}")
if result.returncode != 0:
    print(f"  ⚠️  mcu_compile exited with code {result.returncode}")

---
## ✅ Section 6 — Verify MERA-Compiled Model (py_compute Inference)

Since we used `--host-evaluate` in Section 5.2, the `mcu_compile.py` script has
already built the MERA-generated C source into a `py_compute` shared library
(via CMake with `-DBUILD_PY_BINDINGS=ON`).

This section loads the pre-built `py_compute.so` and runs inference on the sample
images to verify that the compiled model produces correct top-5 predictions
— matching the TFLite INT8 results from Section 4.

> The `py_compute` library is a Python-callable wrapper around the same C-code
> that will run on the RA8P1 board, so matching results here confirms the
> MERA compilation preserved model accuracy.

### ⚠️ Host Build Requirements (for `--host-evaluate`)

| Requirement | Minimum Version | Check Command | Install / Fix |
|-------------|----------------|---------------|---------------|
| **CMake** | ≥ 3.24 | `cmake --version` | `pip install --upgrade cmake` |
| **C++ compiler** (g++) | g++-11 recommended | `g++ --version` | `sudo apt install g++-11` |
| **build-essential** | — | `dpkg -l build-essential` | `sudo apt install build-essential` |

> 💡 **Common issues:**
> - **CMake too old** — Ubuntu 22.04 ships CMake 3.22, but the build needs ≥ 3.24.
>   Upgrade via pip: `pip install --upgrade cmake`
> - **g++ not in PATH** — The cell auto-detects `g++-11`. If that fails, set manually:
>   ```bash
>   export CXX=/usr/bin/g++-11
>   export CC=/usr/bin/gcc-11
>   ```
> - **pybind11** is fetched automatically by CMake — no manual install needed.

In [ ]:
import subprocess, pickle, sys
import numpy as np
import tensorflow as tf
from PIL import Image

# ── Locate the pre-built py_compute.so ──────────────────────────────────────
CPU_BUILD_DIR = EMBED_DIR / "src_mcu"
py_compute_candidates = (
    list(CPU_BUILD_DIR.rglob("py_compute*.so"))
    + list(CPU_BUILD_DIR.rglob("py_compute*.pyd"))
)

if not py_compute_candidates:
    raise FileNotFoundError(
        f"❌ py_compute shared library not found in {CPU_BUILD_DIR}\n"
        f"   Make sure Section 5.2 ran with --host-evaluate."
    )

py_compute_dir = str(py_compute_candidates[0].parent)
print(f"✅ Found py_compute: {py_compute_candidates[0]}")

# ── Read INT8 quantization parameters ────────────────────────────────────────
interp_q = tf.lite.Interpreter(model_path=str(int8_path))
interp_q.allocate_tensors()
_inp_det = interp_q.get_input_details()[0]
_out_det = interp_q.get_output_details()[0]
_inp_qp  = _inp_det["quantization_parameters"]
_out_qp  = _out_det["quantization_parameters"]
quantized_io = {
    "input_scale":  float(_inp_qp["scales"][0]),
    "input_zp":     int(_inp_qp["zero_points"][0]),
    "output_scale": float(_out_qp["scales"][0]),
    "output_zp":    int(_out_qp["zero_points"][0]),
}
del interp_q
print(f"INT8 quant params: inp_scale={quantized_io['input_scale']:.6f}  inp_zp={quantized_io['input_zp']}")
print(f"                   out_scale={quantized_io['output_scale']:.6f}  out_zp={quantized_io['output_zp']}")

# ── Subprocess inference (avoids TF / py_compute symbol conflicts) ───────────
def run_mera_cls_subprocess(py_compute_dir, images, quantized_io):
    config_file = "/tmp/mera_cls_nb_config.pkl"
    result_file = "/tmp/mera_cls_nb_results.pkl"
    with open(config_file, "wb") as f:
        pickle.dump({
            "py_compute_dir": py_compute_dir,
            "images":         images,
            "quantized_io":   quantized_io,
        }, f)

    script = f"""
import sys, pickle
import numpy as np

with open("{config_file}", "rb") as f:
    cfg = pickle.load(f)

sys.path.insert(0, cfg["py_compute_dir"])
import py_compute as c

qio     = cfg["quantized_io"]
results = []
for img in cfg["images"]:
    blob = np.array(img, dtype=np.float32)
    while blob.ndim < 4:
        blob = blob[np.newaxis, ...]
    if qio:
        blob = np.clip(
            np.round(blob / qio["input_scale"] + qio["input_zp"]),
            -128, 127
        ).astype(np.int8)
    raw = c.compute(blob)[0]
    if qio:
        raw = (raw.astype(np.float32) - qio["output_zp"]) * qio["output_scale"]
    elif raw.dtype != np.float32:
        raw = raw.astype(np.float32)
    results.append(raw)

with open("{result_file}", "wb") as f:
    pickle.dump(results, f)
"""
    proc = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"  ❌ Subprocess failed:\n{proc.stderr[:2000]}")
        return None
    with open(result_file, "rb") as f:
        return pickle.load(f)

# ── Load and preprocess sample images ────────────────────────────────────────
# Uses the same sample_images folder and preprocessing as Section 4
SAMPLE_DIR = MODEL_DIR / "python" / "sample_images"
sample_paths = (
    sorted(SAMPLE_DIR.glob("*.jpg"))
    + sorted(SAMPLE_DIR.glob("*.JPEG"))
    + sorted(SAMPLE_DIR.glob("*.png"))
)
if not sample_paths:
    raise FileNotFoundError(f"❌ No sample images found in {SAMPLE_DIR}")

H, W = cfg["input_shape"][:2]
preprocessed = []
for p in sample_paths:
    pil_img = Image.open(p).convert("RGB").resize((W, H))
    net_input = (np.array(pil_img, dtype=np.float32) / 127.5) - 1.0
    preprocessed.append(net_input)

print(f"\nLoaded {len(sample_paths)} sample images from {SAMPLE_DIR}")

# ── TFLite INT8 reference ────────────────────────────────────────────────────
interp_sec6 = tf.lite.Interpreter(model_path=str(int8_path))
interp_sec6.allocate_tensors()
inp_d = interp_sec6.get_input_details()[0]
out_d = interp_sec6.get_output_details()[0]

tflite_results = []
for img in preprocessed:
    blob = img.copy()
    if inp_d["dtype"] == np.int8:
        s, zp = inp_d["quantization_parameters"]["scales"][0], inp_d["quantization_parameters"]["zero_points"][0]
        blob = np.clip(np.round(blob / s + zp), -128, 127).astype(np.int8)
    interp_sec6.set_tensor(inp_d["index"], blob[np.newaxis])
    interp_sec6.invoke()
    raw = interp_sec6.get_tensor(out_d["index"])
    if raw.dtype != np.float32:
        s, zp = out_d["quantization_parameters"]["scales"][0], out_d["quantization_parameters"]["zero_points"][0]
        raw = (raw.astype(np.float32) - zp) * s
    tflite_results.append(raw[0])
del interp_sec6

# ── MERA py_compute ──────────────────────────────────────────────────────────
mera_results = run_mera_cls_subprocess(py_compute_dir, preprocessed, quantized_io)
if mera_results is None:
    raise RuntimeError("MERA inference failed — see error above.")

# ── Compare top-5 predictions ────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  TFLite INT8 vs MERA CPU — Top-5 Prediction Comparison")
print(f"{'='*65}")

top_k = cfg["top_k"]
match_count = 0
for i, p in enumerate(sample_paths):
    tfl_top5 = np.argsort(tflite_results[i])[::-1][:top_k]
    mer_top5 = np.argsort(mera_results[i].flatten())[::-1][:top_k]
    ok = tfl_top5[0] == mer_top5[0]
    match_count += ok
    print(f"\n  {p.name}")
    print(f"    TFLite INT8 top-1: [{tfl_top5[0]:>4}] {label_name(tfl_top5[0])}")
    print(f"    MERA CPU    top-1: [{mer_top5[0]:>4}] {label_name(mer_top5[0])}")
    print(f"    Top-1 match: {'✅' if ok else '❌'}")

print(f"\n  Agreement: {match_count}/{len(sample_paths)} images")
print(f"\n{'='*65}")
print(f"  ✅ MERA CPU model verification complete!")
print(f"{'='*65}")

---
### 📄 Understanding the Generated C Files

MERA produces C source files in `embedded_c/src_mcu_npu/` (NPU) or `embedded_c/src_mcu/` (CPU).
The key files are listed below:

| File | Purpose | Target |
|------|---------|--------|
| `model.c / model.h` | Top-level model entry point — orchestrates subgraph execution. | NPU only |
| `hal_entry.c` | Hardware Abstraction Layer — board-specific init (clocks, Ethos-U driver). | NPU only |
| `sub_XXXX_model_data.c/.h` | Vela-compiled command stream & weight data for NPU subgraphs. | NPU only |
| `sub_XXXX_tensors.c/.h` | Tensor arena allocation for NPU subgraphs. | NPU only |
| `sub_XXXX_command_stream.c/.h` | Ethos-U command stream blobs (compiled by Vela). | NPU only |
| `sub_XXXX_invoke.c/.h` | Invoke helpers that feed command streams to the Ethos-U driver. | NPU only |
| `compute_sub_0000.c/.h` | Main inference graph — kernel calls for each layer (NPU-accelerated or CMSIS-NN). | Both |
| `kernel_library_int.c/.h` | INT8/INT16 kernel implementations (Ethos-U offload or CMSIS-NN). | Both |
| `kernel_library_utils.c/.h` | Shared helpers — memory layout, tensor management, dequantization. | Both |

---

## 🎉 You're Done!

You have successfully completed the full pipeline from a pretrained checkpoint to MCU-ready C source files:

```
✅  Setup       — Configuration
✅  Section 1   — Load pretrained checkpoint
    └─ 1.1 — Verify original Keras model (baseline inference)
✅  Section 2   — Convert to TFLite FP32
✅  Section 3   — Post-training quantization → TFLite INT8
    ├─ 3.1  Calibration dataset generator
    └─ 3.2  INT8 quantization (TFLiteConverter)
✅  Section 4   — Quick inference test (Keras vs FP32 vs INT8)
✅  Section 5   — MERA compilation → embedded C source files
    ├─ 5.1  NPU compilation (Ethos-U55)
    └─ 5.2  CPU-only compilation (CMSIS-NN) + host evaluation build
✅  Section 6   — Verify MERA-compiled CPU model (py_compute inference)
    └─ Compare top-5 predictions vs TFLite INT8 reference
```

The generated C files in `embedded_c/src_mcu/` (and `src_mcu_npu/` for the NPU path) are ready to be integrated into your **e² studio** project and flashed to the **Renesas RA8P1** board.

---